# Notebook 09 -- Grover's Search Algorithm

Classical computers searching an unsorted database of $N$ items need $O(N)$
queries in the worst case -- there is no shortcut when there is no structure
to exploit. Grover's algorithm (1996) achieves a **quadratic speedup**,
finding a marked item in only $O(\sqrt{N})$ queries.

This is provably optimal for unstructured search: no quantum algorithm can
do better than $\Omega(\sqrt{N})$. While not an exponential speedup like
Shor's factoring algorithm, Grover's speedup is universal -- it applies to
**any** search problem where we can check whether a candidate is a solution.

By the end of this notebook you will be able to:

1. **Describe** Grover's algorithm and its quadratic speedup for unstructured search.
2. **Implement** phase and boolean oracles for marking target states.
3. **Predict** the optimal number of Grover iterations for a given search space.
4. **Explain** why too many iterations causes over-rotation and reduces success probability.

In this notebook we will:

1. Implement Grover's algorithm using the Goqu SDK.
2. Understand the **oracle** and **diffusion operator**.
3. Explore the geometric picture of **amplitude amplification**.
4. See what happens when we iterate too many times (**over-rotation**).
5. Search for multiple solutions simultaneously.

In [ ]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/grover"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## The Problem: Searching Without Structure

Imagine you have a phone book with $N = 2^n$ entries, but the entries are
in random order. You want to find the entry matching a specific criterion.
Classically, you must check entries one by one -- on average $N/2$ queries.

In the quantum setting, we encode the search space into $n$ qubits, giving
us $N = 2^n$ computational basis states. A **quantum oracle** marks the
target state(s) by flipping their phase. Grover's algorithm then amplifies
the amplitude of the marked states until they dominate the measurement
outcome.

| Qubits ($n$) | Search space ($N$) | Classical queries | Grover queries |
|:---:|:---:|:---:|:---:|
| 3 | 8 | 4 | 2 |
| 4 | 16 | 8 | 3 |
| 10 | 1024 | 512 | 25 |
| 20 | 1,048,576 | 524,288 | 804 |

## How Grover's Algorithm Works

Grover's algorithm repeats two operations in a loop:

1. **Oracle** $O$: Flips the phase of the target state(s). If $|x\rangle$
   is a target, $O|x\rangle = -|x\rangle$; otherwise, $O|x\rangle = |x\rangle$.

2. **Diffusion operator** $D = 2|s\rangle\langle s| - I$: Reflects all
   amplitudes about their mean, where $|s\rangle = H^{\otimes n}|0\rangle$
   is the uniform superposition.

### Geometric Picture

The state lives in a 2D plane spanned by the target state $|t\rangle$
and the uniform superposition of non-target states $|t^\perp\rangle$.
Each Grover iteration rotates the state vector by angle
$2\theta$ toward $|t\rangle$, where $\sin\theta = \sqrt{M/N}$
($M$ = number of solutions).

The initial state $|s\rangle$ has a small angle $\theta$ to
$|t^\perp\rangle$. After $k$ iterations the angle becomes
$(2k+1)\theta$. We want this to be as close to $\pi/2$ as possible
(perfectly aligned with $|t\rangle$), giving the optimal iteration count:

$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4} \sqrt{\frac{N}{M}} \right\rfloor$$

### Building Grover's algorithm from scratch

Before using the SDK, let's build one iteration manually on 3 qubits to
see exactly what happens. The two components are:

1. **Oracle** -- flips the phase of the target state |101>
2. **Diffusion** -- reflects about the mean amplitude (H-X-CCZ-X-H)

In [ ]:
%%
// Manual Grover on 3 qubits, searching for |101>
b := builder.New("grover-manual", 3)

// Step 1: Initialize uniform superposition
b.H(0).H(1).H(2)

// === One Grover iteration ===

// Step 2: Oracle — flip phase of |101> (q0=1, q1=0, q2=1)
// To mark |101>: apply X to q1 (flip the 0-bit), then CCZ, then X to q1
b.X(1)
b.CCZ(0, 1, 2)
b.X(1)

// Step 3: Diffusion operator (reflect about |+...+>)
b.H(0).H(1).H(2)
b.X(0).X(1).X(2)
b.CCZ(0, 1, 2)
b.X(0).X(1).X(2)
b.H(0).H(1).H(2)

// Measure
b.MeasureAll()
c, _ := b.Build()

fmt.Println("Manual Grover circuit (1 iteration):")
fmt.Println(draw.String(c))

sim := statevector.New(3)
counts, _ := sim.Run(c, 1000)
fmt.Println("\nCounts:", counts)
gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("Manual Grover: 1 iteration for |101>")))

The SDK's `grover.Run` handles oracle construction, optimal iteration count, and measurement for you:

## Basic Grover Search

Let's search for the state $|1010\rangle$ (decimal 10) in a 4-qubit space
of $N = 16$ items. With 1 solution, the optimal iteration count is
$\lfloor \pi/4 \cdot \sqrt{16} \rfloor = \lfloor \pi \rfloor = 3$.

In [ ]:
%%
ctx := context.Background()

// Search for |1010> (decimal 10) in a 4-qubit search space
result, err := grover.Run(ctx, grover.Config{
	NumQubits:    4,
	Oracle:       grover.PhaseOracle([]int{0b1010}, 4),
	NumSolutions: 1,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Grover Search: 4 qubits, target |1010> ===")
fmt.Println("Top result:", result.TopResult)
fmt.Println("Iterations used:", result.NumIters)
fmt.Println("Measurement counts:", result.Counts)

// Calculate success probability
total := 0
for _, v := range result.Counts {
	total += v
}
if count, ok := result.Counts["1010"]; ok {
	fmt.Printf("Success probability: %.1f%%\n", float64(count)/float64(total)*100)
}

In [ ]:
%%
ctx := context.Background()

result, _ := grover.Run(ctx, grover.Config{
	NumQubits:    4,
	Oracle:       grover.PhaseOracle([]int{0b1010}, 4),
	NumSolutions: 1,
	Shots:        1000,
})

fmt.Println("Circuit diagram:")
fmt.Println(draw.String(result.Circuit))

svg := viz.Histogram(result.Counts, viz.WithTitle("Grover Search: target |1010>"))
gonbui.DisplayHTML(svg)

## Phase Oracle vs Boolean Oracle

The Goqu SDK provides two ways to construct an oracle:

**Phase Oracle** (`grover.PhaseOracle`): You specify the target states
directly as integer indices. This is the most efficient approach when you
know the targets ahead of time.

**Boolean Oracle** (`grover.BooleanOracle`): You provide a classical
function $f(x) \to \text{bool}$ that returns `true` for target states.
The oracle internally evaluates $f$ on all inputs to find the targets.
This is more natural for constraint-satisfaction problems where the
"target" is defined by a property rather than an explicit list.

Both produce the same phase-kickback effect: marked states get a $-1$
phase flip.

In [ ]:
%%
ctx := context.Background()

// Phase oracle: directly specify target |101> = 5
phaseOracle := grover.PhaseOracle([]int{0b101}, 3)

// Boolean oracle: define a function that returns true for target
boolOracle := grover.BooleanOracle(func(x int) bool {
	return x == 0b101
}, 3)

// Run both and compare
resultPhase, _ := grover.Run(ctx, grover.Config{
	NumQubits:    3,
	Oracle:       phaseOracle,
	NumSolutions: 1,
	Shots:        1000,
})

resultBool, _ := grover.Run(ctx, grover.Config{
	NumQubits:    3,
	Oracle:       boolOracle,
	NumSolutions: 1,
	Shots:        1000,
})

fmt.Println("=== Phase Oracle ===")
fmt.Println("Top result:", resultPhase.TopResult)
fmt.Println("Counts:", resultPhase.Counts)

fmt.Println()
fmt.Println("=== Boolean Oracle ===")
fmt.Println("Top result:", resultBool.TopResult)
fmt.Println("Counts:", resultBool.Counts)

fmt.Println()
fmt.Println("Both oracles find the same target -- they produce equivalent circuits.")

svgPhase := viz.Histogram(resultPhase.Counts, viz.WithTitle("Phase Oracle: target |101>"))
svgBool := viz.Histogram(resultBool.Counts, viz.WithTitle("Boolean Oracle: target |101>"))
gonbui.DisplayHTML(svgPhase)
gonbui.DisplayHTML(svgBool)

## Optimal Number of Iterations

The magic of Grover's algorithm lies in the **amplitude amplification**
process. Each iteration rotates the state vector closer to the target
subspace. The optimal number of iterations is:

$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4} \sqrt{\frac{N}{M}} \right\rfloor$$

where $N = 2^n$ is the search space size and $M$ is the number of solutions.

At the optimal iteration count, the probability of measuring a target state
is close to 1. With fewer or more iterations, the probability is lower.

Let's visualize how the success probability changes with the number of
iterations for a 3-qubit search (N=8, M=1).

In [ ]:
%%
ctx := context.Background()
n := 3
N := 1 << n // 8
M := 1

optIter := int(math.Floor(math.Pi / 4 * math.Sqrt(float64(N)/float64(M))))
fmt.Printf("Search space: N = %d, solutions: M = %d\n", N, M)
fmt.Printf("Optimal iterations: floor(pi/4 * sqrt(%d/%d)) = %d\n\n", N, M, optIter)

// Run Grover with different iteration counts and track success probability
probCounts := make(map[string]int)
for iters := 1; iters <= 6; iters++ {
	result, _ := grover.Run(ctx, grover.Config{
		NumQubits: n,
		Oracle:    grover.PhaseOracle([]int{0b101}, n),
		NumIters:  iters,
		Shots:     2000,
	})

	// Calculate success probability
	total := 0
	for _, v := range result.Counts {
		total += v
	}
	successes := 0
	if c, ok := result.Counts["101"]; ok {
		successes = c
	}
	prob := float64(successes) / float64(total) * 100

	marker := ""
	if iters == optIter {
		marker = " <-- optimal"
	}
	fmt.Printf("Iterations: %d  |  P(target) = %5.1f%%%s\n", iters, prob, marker)

	// Build a histogram-friendly map: label = iteration count, value = successes
	label := fmt.Sprintf("k=%d", iters)
	probCounts[label] = successes
}

fmt.Println("\nThe success probability peaks at the optimal iteration count.")
fmt.Println("Too few iterations: not enough amplification.")
fmt.Println("Too many iterations: the state rotates past the target (over-rotation).")

svg := viz.Histogram(probCounts, viz.WithTitle("Success Count vs Iteration Count (3 qubits, 2000 shots)"))
gonbui.DisplayHTML(svg)

## Over-Rotation

A crucial subtlety: Grover's algorithm is **not** "the more iterations,
the better." Because the state rotates in a 2D plane, iterating past the
optimal point causes the state to **rotate away** from the target.

After the optimal number of iterations, additional iterations decrease
the success probability. The state oscillates sinusoidally, periodically
returning to near-zero probability of finding the target.

This is fundamentally different from classical search, where checking
more items never hurts. In Grover's algorithm, **knowing when to stop**
is essential.

In [ ]:
%%
ctx := context.Background()

// Demonstrate over-rotation on 3 qubits with 10 iterations
// Optimal is 2 iterations; 10 is way past the sweet spot
fmt.Println("=== Over-Rotation Demo: 3 qubits, target |110> ===")
fmt.Println()

for _, iters := range []int{1, 2, 3, 5, 7, 10} {
	result, _ := grover.Run(ctx, grover.Config{
		NumQubits: 3,
		Oracle:    grover.PhaseOracle([]int{0b110}, 3),
		NumIters:  iters,
		Shots:     2000,
	})

	total := 0
	for _, v := range result.Counts {
		total += v
	}
	successes := 0
	if c, ok := result.Counts["110"]; ok {
		successes = c
	}
	prob := float64(successes) / float64(total) * 100

	bar := ""
	for j := 0; j < int(prob/2); j++ {
		bar += "#"
	}
	fmt.Printf("k=%2d : P(target) = %5.1f%% %s\n", iters, prob, bar)
}

fmt.Println()
fmt.Println("The probability oscillates! At k=2 it peaks near 95%,")
fmt.Println("but at k=5 it drops below 1%, then rises again around k=7-8.")
fmt.Println("This periodic behavior is a direct consequence of the rotation picture.")

## Multiple Solutions

Grover's algorithm generalizes naturally to searching for $M > 1$ marked
items. With more solutions, fewer iterations are needed because the target
subspace is larger (the initial overlap $\sin\theta = \sqrt{M/N}$ is
bigger).

The optimal iteration count becomes:

$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4} \sqrt{\frac{N}{M}} \right\rfloor$$

For example, with $N = 8$ and $M = 2$: $k_{\text{opt}} = \lfloor \pi/4 \cdot 2 \rfloor = 1$.

The measurement outcome will be one of the marked states, chosen
approximately uniformly at random.

In [ ]:
%%
ctx := context.Background()

// Search for two targets: |011> = 3 and |110> = 6
result, err := grover.Run(ctx, grover.Config{
	NumQubits:    3,
	Oracle:       grover.PhaseOracle([]int{0b011, 0b110}, 3),
	NumSolutions: 2,
	Shots:        2000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Multiple Solutions: targets |011> and |110> ===")
fmt.Println("Top result:", result.TopResult)
fmt.Println("Iterations used:", result.NumIters)
fmt.Println("Counts:", result.Counts)

// Show the counts for the two targets
total := 0
for _, v := range result.Counts {
	total += v
}
for _, target := range []string{"011", "110"} {
	if c, ok := result.Counts[target]; ok {
		fmt.Printf("  |%s> : %d (%.1f%%)\n", target, c, float64(c)/float64(total)*100)
	}
}

fmt.Println("\nBoth targets are amplified roughly equally.")

svg := viz.Histogram(result.Counts, viz.WithTitle("Grover: Two Solutions |011> and |110>"))
gonbui.DisplayHTML(svg)

## Predict, Then Verify

**Question:** How many Grover iterations are optimal for searching a
4-qubit space ($N = 16$) with exactly 1 solution?

**Prediction:** Using the formula:

$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4} \sqrt{\frac{16}{1}} \right\rfloor = \left\lfloor \frac{\pi}{4} \cdot 4 \right\rfloor = \left\lfloor \pi \right\rfloor = 3$$

Let's verify that Grover's algorithm automatically uses 3 iterations.

In [ ]:
%%
ctx := context.Background()

// Let the algorithm choose the optimal iteration count (NumIters = 0)
result, err := grover.Run(ctx, grover.Config{
	NumQubits:    4,
	Oracle:       grover.PhaseOracle([]int{0b1100}, 4),
	NumSolutions: 1,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

expected := int(math.Floor(math.Pi / 4 * math.Sqrt(16.0/1.0)))
fmt.Printf("Predicted optimal iterations: %d\n", expected)
fmt.Printf("Actual iterations used:       %d\n", result.NumIters)
fmt.Printf("Match: %v\n", result.NumIters == expected)
fmt.Println("\nTop result:", result.TopResult)
fmt.Println("Counts:", result.Counts)

---

## Exercises

### Exercise 1 -- Search for Multiple Solutions

Use Grover's algorithm on 4 qubits to search for three targets simultaneously:
$|0011\rangle$, $|0110\rangle$, and $|1001\rangle$.

Before running the code, predict the optimal iteration count using the formula
$\lfloor \pi/4 \cdot \sqrt{16/3} \rfloor$. Then verify by checking `result.NumIters`.

In [ ]:
%%
// Exercise 1: Search for 3 targets in a 4-qubit space
// Targets: |0011>, |0110>, |1001>
//
// Step 1: Predict the optimal iteration count using the formula
//         floor(pi/4 * sqrt(N/M)) where N=16, M=3
//
// Step 2: Run Grover's algorithm and verify your prediction
//
// TODO: Calculate and print the predicted iteration count
// predicted := int(math.Floor(math.Pi / 4 * math.Sqrt(???)))
// fmt.Printf("Predicted optimal iterations: %d\n\n", predicted)
//
// TODO: Configure and run Grover's search for 3 targets
// result, err := grover.Run(ctx, grover.Config{
//     NumQubits:    ???,
//     Oracle:       grover.PhaseOracle([]int{???}, ???),
//     NumSolutions: ???,
//     Shots:        2000,
// })
//
// TODO: Print iterations used, top result, and counts
// TODO: Calculate and print the success probability for each target
//
// TODO: Visualize with viz.Histogram
ctx := context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

### Exercise 2 -- Custom Boolean Oracle for a SAT-like Problem

Use `grover.BooleanOracle` to search for all 3-bit numbers where
**bit 0 AND bit 1 are both 1** (i.e., the two least-significant bits
are set). The satisfying states are $|011\rangle = 3$ and
$|111\rangle = 7$.

This demonstrates how Grover's algorithm can solve constraint-satisfaction
problems: you express the constraint as a boolean function and let the
algorithm find the satisfying assignments.

In [ ]:
%%
// Exercise 2: Boolean oracle for a SAT-like constraint
// Find all 3-bit numbers x where bit0 AND bit1 == 1
// (both least-significant bits are set)
// Satisfying states: |011> = 3 and |111> = 7
//
// TODO: Create a BooleanOracle that checks whether both bit0 and bit1 are 1
// oracle := grover.BooleanOracle(func(x int) bool {
//     bit0 := ???
//     bit1 := ???
//     return ???
// }, 3)
//
// TODO: Run Grover's search with NumSolutions=2 on 3 qubits
// result, err := grover.Run(ctx, grover.Config{
//     NumQubits:    ???,
//     Oracle:       oracle,
//     NumSolutions: ???,
//     Shots:        2000,
// })
//
// TODO: Print the top result, iterations used, and measurement counts
// TODO: Print the success probability for each satisfying state (011, 111)
// TODO: Visualize with viz.Histogram
ctx := context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

---

## Key Takeaways

1. **Grover's algorithm** provides a provably optimal quadratic speedup
   for unstructured search: $O(\sqrt{N})$ queries instead of $O(N)$.

2. The algorithm alternates two operations: an **oracle** that marks target
   states with a phase flip, and a **diffusion operator** that amplifies
   marked amplitudes by reflecting about the mean.

3. **Geometrically**, each iteration rotates the state vector by $2\theta$
   toward the target subspace. The optimal number of iterations is
   $\lfloor \pi/4 \cdot \sqrt{N/M} \rfloor$.

4. **Over-rotation** is a real danger: too many iterations cause the state
   to rotate past the target, reducing the success probability. Unlike
   classical search, more work can make things worse.

5. **Phase oracles** mark targets by index; **boolean oracles** mark
   targets by evaluating a classical predicate. Both are equivalent in
   effect, but boolean oracles are more natural for constraint-satisfaction
   and SAT-like problems.

6. With $M$ multiple solutions, the algorithm needs fewer iterations
   ($\sqrt{N/M}$ instead of $\sqrt{N}$) and returns one of the solutions
   uniformly at random.

---

**Next up:** Notebook 10, where we build on these foundations to explore
more advanced quantum algorithms.